# Practical 8 – Principal Component Analysis (PCA)
**Topics:** Explained Variance, Image Compression, MNIST, Olivetti Faces

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_openml, fetch_olivetti_faces
from sklearn.preprocessing import StandardScaler

---
## Task 1 – Dimensions vs Explained Variance (0.5-division x-axis)

We load MNIST, fit PCA with increasing numbers of components, and plot cumulative explained variance.
The twist: the x-axis (dimensions) is scaled so each tick represents **0.5 dimensions** instead of 1.

In [ ]:
# Load MNIST (70 000 samples, 784 features)
print('Loading MNIST ...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_mnist = mnist['data'].astype(np.float32)
y_mnist = mnist['target'].astype(int)
print(f'Shape: {X_mnist.shape}')

In [ ]:
# Fit PCA once to get ALL explained variance ratios efficiently
pca_full = PCA(n_components=784, svd_solver='full')
pca_full.fit(X_mnist)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components_range = np.arange(1, 785)

In [ ]:
# ── Task 1: x-axis in 0.5-dimension divisions ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(n_components_range, cumvar, linewidth=1.5, color='steelblue')
ax.axhline(0.95, color='tomato',   linestyle='--', linewidth=1, label='95% variance')
ax.axhline(0.99, color='darkorange', linestyle='--', linewidth=1, label='99% variance')

# --- stretch x-axis: set ticks every 0.5 unit on the DISPLAYED axis --------
# We keep the data in real component counts (1-784) but relabel the ticks
# so that each major tick represents 0.5 'displayed' dimensions.
# Equivalently: display = n_components / 2  →  for every real component n,
# the displayed position is n/2.

# Replot on a new x-axis scaled by 0.5
ax.cla()
x_display = n_components_range / 2           # each real component → 0.5 on display
ax.plot(x_display, cumvar, linewidth=1.5, color='steelblue', label='Cumulative Explained Variance')
ax.axhline(0.95, color='tomato',     linestyle='--', linewidth=1, label='95% threshold')
ax.axhline(0.99, color='darkorange', linestyle='--', linewidth=1, label='99% threshold')

# Ticks every 0.5 on the display axis  →  labels show actual component count
tick_display = np.arange(0, x_display[-1] + 0.5, 0.5)   # 0, 0.5, 1.0, …
tick_labels  = (tick_display * 2).astype(int)             # 0, 1, 2, …

# Only show a sensible subset to avoid clutter (every 20 real components)
keep = tick_labels % 20 == 0
ax.set_xticks(tick_display[keep])
ax.set_xticklabels(tick_labels[keep], fontsize=8)
ax.xaxis.set_minor_locator(plt.MultipleLocator(0.5))     # minor tick every 0.5

ax.set_xlabel('Number of Dimensions (each division = 0.5 dim)', fontsize=11)
ax.set_ylabel('Cumulative Explained Variance Ratio', fontsize=11)
ax.set_title('Task 1 – MNIST: Dimensions vs Explained Variance\n(x-axis in 0.5-dimension divisions)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('task1_dims_vs_variance.png', dpi=150)
plt.show()
print('Saved: task1_dims_vs_variance.png')

---
## Task 2 – Compress an Image Retaining 95% Explained Variance

We apply PCA to the Olivetti faces image (as done in the lecture slides around slide 130),
but set `n_components` so that **95% of the total variance** is retained.

In [ ]:
# Load Olivetti faces (same dataset used in the lecture compression demo)
print('Loading Olivetti faces ...')
faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X_faces = faces.data          # shape (400, 4096) — 64×64 images
print(f'Shape: {X_faces.shape}')

In [ ]:
# ── Task 2: PCA retaining 95% explained variance ───────────────────────────
# Passing a float in (0, 1) to n_components tells sklearn to automatically
# select the minimum number of components to reach that explained variance.
pca_95 = PCA(n_components=0.95, svd_solver='full')
X_compressed_95 = pca_95.fit_transform(X_faces)
X_reconstructed_95 = pca_95.inverse_transform(X_compressed_95)

n_kept = pca_95.n_components_
total_var = pca_95.explained_variance_ratio_.sum()
print(f'Components kept for 95% variance: {n_kept} / 4096')
print(f'Actual variance retained: {total_var:.4f}')

In [ ]:
n_show = 5
fig, axes = plt.subplots(2, n_show, figsize=(14, 6))

for i in range(n_show):
    # Original
    axes[0, i].imshow(X_faces[i].reshape(64, 64), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Original', fontsize=9)
    axes[0, i].axis('off')
    # Compressed (95% variance)
    axes[1, i].imshow(X_reconstructed_95[i].reshape(64, 64), cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'95% var\n({n_kept} dims)', fontsize=9)
    axes[1, i].axis('off')

fig.suptitle('Task 2 – Olivetti Faces: Original vs PCA Compressed (95% Variance Retained)', fontsize=12)
plt.tight_layout()
plt.savefig('task2_compressed_95pct.png', dpi=150)
plt.show()
print('Saved: task2_compressed_95pct.png')

---
## Task 3 – Compressing MNIST (10×10 grid)
### (a) Original 784 dimensions  |  (b) 90% explained variance

In [ ]:
# Use the first 100 MNIST samples
X_100 = X_mnist[:100]   # shape (100, 784)

def plot_10x10(images, title, filename, img_shape=(28, 28), cmap='binary'):
    """Plot 100 images in a 10×10 grid."""
    fig, axes = plt.subplots(10, 10, figsize=(10, 10))
    for idx, ax in enumerate(axes.flat):
        ax.imshow(images[idx].reshape(img_shape), cmap=cmap, vmin=0, vmax=255 if images.max() > 1 else 1)
        ax.axis('off')
    fig.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filename}')

In [ ]:
# (a) Original – all 784 dimensions
plot_10x10(
    X_100,
    title='Task 3(a) – First 100 MNIST Digits: Original (784 dimensions)',
    filename='task3a_mnist_original.png'
)

In [ ]:
# (b) PCA with 90% explained variance, then reconstruct
pca_90 = PCA(n_components=0.90, svd_solver='full')
pca_90.fit(X_mnist)                                       # fit on full dataset
X_100_reduced   = pca_90.transform(X_100)                 # encode
X_100_reconstructed = pca_90.inverse_transform(X_100_reduced)  # decode

n_dims_90 = pca_90.n_components_
print(f'Components for 90% variance: {n_dims_90}')
print(f'Compression ratio: {784/n_dims_90:.1f}×  ({n_dims_90} → 784 features)')

plot_10x10(
    X_100_reconstructed,
    title=f'Task 3(b) – First 100 MNIST Digits: 90% Variance ({n_dims_90} dimensions)',
    filename='task3b_mnist_90pct.png'
)

---
## Task 4 – Optimal Dimensions for Olivetti Faces via Explained Variance

We replicate the elbow/knee method from Slides 103-110 (applied to MNIST)
but now on the **Olivetti faces** dataset to find the approximate optimal number of PCA dimensions.

In [ ]:
# Fit PCA across all possible components for Olivetti (max = min(400 samples, 4096 features) = 400)
pca_faces_full = PCA(n_components=399, svd_solver='full')
pca_faces_full.fit(X_faces)

cumvar_faces = np.cumsum(pca_faces_full.explained_variance_ratio_)
n_range_faces = np.arange(1, 400)

In [ ]:
# Find dimension thresholds
thresh_90 = np.searchsorted(cumvar_faces, 0.90) + 1
thresh_95 = np.searchsorted(cumvar_faces, 0.95) + 1
thresh_99 = np.searchsorted(cumvar_faces, 0.99) + 1
print(f'Components for 90% variance: {thresh_90}')
print(f'Components for 95% variance: {thresh_95}')
print(f'Components for 99% variance: {thresh_99}')

In [ ]:
# ── Task 4: Plot explained variance curve with knee/elbow analysis ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Individual explained variance (scree plot) ---
axes[0].plot(n_range_faces, pca_faces_full.explained_variance_ratio_,
             color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Number of Components', fontsize=11)
axes[0].set_ylabel('Individual Explained Variance Ratio', fontsize=11)
axes[0].set_title('Scree Plot – Olivetti Faces', fontsize=12)
axes[0].grid(True, alpha=0.3)

# --- Right: Cumulative explained variance with thresholds ---
axes[1].plot(n_range_faces, cumvar_faces, color='steelblue', linewidth=2,
             label='Cumulative Explained Variance')

for thresh, pct, col in [(thresh_90, '90%', 'green'),
                          (thresh_95, '95%', 'tomato'),
                          (thresh_99, '99%', 'darkorange')]:
    axes[1].axhline(float(pct[:-1])/100, color=col, linestyle='--', linewidth=1, alpha=0.7)
    axes[1].axvline(thresh, color=col, linestyle=':', linewidth=1.5,
                    label=f'{pct} → {thresh} dims')
    axes[1].annotate(f'{thresh}', xy=(thresh, float(pct[:-1])/100),
                     xytext=(thresh + 5, float(pct[:-1])/100 - 0.04),
                     fontsize=9, color=col,
                     arrowprops=dict(arrowstyle='->', color=col, lw=1))

axes[1].set_xlabel('Number of Components', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=11)
axes[1].set_title('Task 4 – Optimal Dimensions: Olivetti Faces', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task4_olivetti_optimal_dims.png', dpi=150)
plt.show()
print('Saved: task4_olivetti_optimal_dims.png')
print(f'\n→ Approximate optimal dimensions for Olivetti faces:')
print(f'   90% variance → {thresh_90} components')
print(f'   95% variance → {thresh_95} components  ← typical choice')
print(f'   99% variance → {thresh_99} components')

### Summary

| Task | Key Result |
|------|------------|
| 1 | Cumulative variance vs dimensions plot with 0.5-division x-axis |
| 2 | Olivetti faces compressed with PCA retaining 95% variance |
| 3(a) | First 100 MNIST digits at original 784 dimensions |
| 3(b) | First 100 MNIST digits reconstructed after 90%-variance PCA |
| 4 | Elbow analysis on Olivetti faces; ~95% threshold ≈ 200–250 components |